In [ ]:
%cd ../../..

In [ ]:
import pathlib

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import torch
from torch import nn

from IPython.display import HTML, display
from pprint import pprint


import src.visuals as visual
import src.utils as utils

from src.models import PINN
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_dataloaders
from src.train import train_model

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)


frac_size = 0.05

file_name = "data"
data_path = pathlib.Path(f"data/real_data/frac_{int(100*frac_size)}")

train_df, valid_df, test_df = load_data(data_path, file_name)

input_col_names = ['time', 're', 'x', 'y']
target_col_names = ['U_x', 'U_y', 'p']

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
train_df.describe()

In [ ]:
valid_df.describe()

In [ ]:
test_df.describe()

In [ ]:
mean = train_df.mean()
std = train_df.std()

train_df = utils.normalize_data(train_df, mean, std)
valid_df = utils.normalize_data(valid_df, mean, std)
test_df = utils.normalize_data(test_df, mean, std)

In [ ]:
train_dataloader, valid_dataloader, test_dataloader = gen_dataloaders(train_df, 
                                                                      valid_df, 
                                                                      test_df, 
                                                                      input_col_names, 
                                                                      target_col_names,
                                                                      32768)

In [ ]:
model = PINN(len(input_col_names), len(target_col_names)).to(device)
criterion = NavierStokesLoss(1, mean, std)
optimizer = torch.optim.Adam(model.parameters())

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=15
)

EPOCHS = 500

In [ ]:
n_param = sum([p.numel() for p in model.parameters()])

print("Number of parameters: ", n_param)

In [ ]:
run_dir = utils.create_run_directory(frac_size, label="cphys_1")

history = train_model(
    model,
    train_dataloader,
    valid_dataloader,
    criterion,
    optimizer,
    scheduler,
    device,
    EPOCHS,
    run_dir,
    checkpoint=None,
    physics_loss=True,
)

In [ ]:
def plot(df, ax):
    for x in ax:
        x.ticklabel_format(axis='y', style='sci', scilimits=(-2,2), useMathText=True)

    df[["train", "valid"]].plot(ax=ax[0])
    df[["train_data", "valid_data"]].plot(ax=ax[1])
    df[["train_physics", "valid_physics"]].plot(ax=ax[2])
    
history_df = pd.DataFrame(history)

fig, ax = plt.subplots(4, 3, figsize=(12, 12), tight_layout=True)

plot(history_df, ax[0])
plot(history_df[:100], ax[1])
plot(history_df[100:300], ax[2])
plot(history_df[300:], ax[3])

fig.savefig(run_dir / "losses.png")
history_df.to_csv(run_dir / "losess.csv")

In [ ]:
# Učitaj originalne podatke za vizuelizaciju i evaluaciju
data_path = pathlib.Path(f"data/real_data/frac_{int(100*frac_size)}")
file_name = "data"

train_df_original, valid_df_original, test_df_original = load_data(data_path, file_name)


train_df_original = train_df_original.sort_values(
    ["re", "time", "y", "x"]
).reset_index(drop=True)

valid_df_original = valid_df_original.sort_values(
    ["re", "time", "y", "x"]
).reset_index(drop=True)

test_df_original = test_df_original.sort_values(
    ["re", "time", "y", "x"]
).reset_index(drop=True)

print(f"Train skup: {train_df_original.shape[0]} redova")
print(f"Valid skup: {valid_df_original.shape[0]} redova")
print(f"Test skup: {test_df_original.shape[0]} redova")
print(f"\nVremenske korake: {sorted(train_df_original['time'].unique().tolist())}")
print(f"Reynolds brojevi: {sorted(train_df_original['re'].unique().tolist())}")

## Vizuelizacija

In [ ]:
# Primjer: vizuelizacija test skupa
re_values = sorted(test_df_original['re'].unique().tolist())
time_steps = sorted(test_df_original['time'].unique().tolist())

print(f"Dostupni Reynolds brojevi: {re_values[:5]}... (ukupno {len(re_values)})")
print(f"Dostupni vremenski koraci: {time_steps}")

# Vizuelizacija jednog Rejnoldsovog broja u datom vremenskom trenutku
visual.plot_velocity_and_pressure(test_df_original, time_steps[5], re_values[5], "Test:")

In [ ]:
# Vizuelizacija evolucije kroz test skupa za jedan Rejnoldsov broj
visual.plot_evolution_in_time(test_df_original, re_values[5])

In [ ]:
# Poređenje predviđanja modela sa stvarnim podacima
visual.compare_predictions(model, test_df_original, time_steps[3], re_values[5], mean, std, device)

In [ ]:
visual.evaluate(model, test_df_original, mean, std, device, run_dir / "animations", 2)

In [ ]:
visual.animate_truth_vs_pred(model, test_df_original, re_values[5], 
                             mean, std, device, fps=2, 
                             output_file=run_dir / "animations" / "truth_vs_pred.gif")

## Evaluacija

In [ ]:
# Full evaluation summary for all splits

split_dfs = {
    "valid": valid_df_original,
    "test": test_df_original,
}

for split_name, split_df in split_dfs.items():
    print(f"\n{'=' * 60}")
    print(f"{split_name.upper()} EVALUATION")
    print(f"{'=' * 60}")

    data_metrics = utils.evaluate_model(
        model=model,
        df=split_df,
        input_col_names=input_col_names,
        target_col_names=target_col_names,
        mean=mean,
        std=std,
        device=device,
    )

    print(f"\nSamples: {len(split_df)}")
    print("\nSupervised metrics (original scale):")
    pprint(data_metrics)

In [ ]:
import importlib
importlib.reload(visual)